<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Why Numerical Methods? Floating Point, Error & Complexity</div>
</div>

#### This notebook introduces the computational foundations that sit underneath every numerical and statistical technique you'll use in this course. 
***

## <span style="color:#B5121B">1. Computers don't do algebra</span>

A computer doesn't manipulate symbols the way you do on paper. It does arithmetic on a finite-precision approximation of a real number, many times, very fast. That distinction sounds pedantic until you see it break.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

As in the lecture, `0.1` and `0.2` cannot be represented *exactly* in binary floating point (just as $1/3$ cannot be written exactly in decimal). Each is stored as the closest double-precision value, and the small representation errors don't cancel when you add them. This isn't a Python bug: it is IEEE 754 double precision, the standard used by almost every numerical language.

Floating-point addition is also not strictly associative:

In [ ]:
a = 1e16
b = -1e16
c = 1.0

print((a + b) + c)   # cancel the two huge numbers first, then add the small one
print(a + (b + c))   # add the small number to a huge one first

Same three numbers, same `+`, different grouping, different answer. `b + c` rounds straight back to `b`, because `1.0` is too small to survive being added to $-10^{16}$ in double precision. The lesson isn't "avoid floating point". It is that **the order of operations in your code can change the answer.**

> **Quick check:** find a fourth number `d` such that `a + b + c + d` gives yet another answer depending on the grouping. (There is more than one way to do this.)
***

## <span style="color:#B5121B">2. Catastrophic cancellation: when a textbook-correct formula fails</span>

Suppose a detector logs the arrival time of every event, in seconds since the experiment started. After about 17 days the timestamps are around 1.5 million seconds, but what you care about is the *spread* (jitter) between events, which might be only a millisecond.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
offset = 1.5e6          # seconds since experiment start (~17 days)
true_jitter_std = 1e-3  # true timing jitter: 1 millisecond

timestamps = offset + rng.normal(0, true_jitter_std, size=100_000)
print("first few timestamps:", timestamps[:3])
print("true variance (by construction):", true_jitter_std ** 2)

The lecture introduced the sum-of-squares formula for the variance, $\mathrm{Var}(x) = E[x^2] - (E[x])^2$. Compute it that way and compare with NumPy's `np.var`:

In [ ]:
naive_var = np.mean(timestamps**2) - np.mean(timestamps)**2
numpy_var = np.var(timestamps)

print(f"naive  E[x^2]-E[x]^2 : {naive_var: .6e}")
print(f"numpy  np.var(x)     : {numpy_var: .6e}")
print(f"true value           : {true_jitter_std**2: .6e}")

**The naive formula gives a *negative* variance.** A variance is a sum of squares, so it is mathematically _impossible_ for it to be negative. When your result suggests an impossible number, your first thought should be that something in the *computation* is broken, not that the underlying physics is wrong.

Why? `timestamps**2` is around $(1.5\times10^6)^2 = 2.25\times10^{12}$, and double precision carries about 15–17 significant figures. The millisecond jitter sits about nine orders of magnitude below that. `E[x^2]` and `(E[x])^2` are two huge, nearly equal numbers: subtracting them removes all the useful digits and leaves rounding noise. This is the **catastrophic cancellation** from the lecture, and it is one of the most common ways scientific code silently gives the wrong answer.

`np.var` avoids this. It computes the mean first and then works with the deviations `x - mean(x)`, which are on the scale of the spread rather than the offset. Same quantity, same data, different order of operations.

### A stable, one-pass alternative: Welford's algorithm

If you can't call `np.var` (for example, if measurements arrive one at a time and you can't store them all), use Welford's algorithm (1962) from the lecture. It updates the mean and variance one value at a time and never squares the large numbers.

In [ ]:
def welford_variance(data):
    n = 0
    mean = 0.0
    M2 = 0.0
    for x in data:
        n += 1
        delta = x - mean
        mean += delta / n
        delta2 = x - mean
        M2 += delta * delta2
    return M2 / n

welford_var = welford_variance(timestamps)
print(f"Welford's algorithm  : {welford_var: .6e}")
print(f"true value           : {true_jitter_std**2: .6e}")

This agrees with NumPy and with the true value to several significant figures. The formula on the page and the algorithm in your code are not the same thing.

> **Exercise:** try two or three `(offset, true_jitter_std)` combinations of your own, with offsets much larger and much smaller than $1.5\times10^6$. For each, note whether the naive formula gives a wrong but plausible number, exactly zero, or a negative number. Does `np.var` ever get it wrong? Can you find a combination where *even Welford's algorithm* struggles?
***

## <span style="color:#555656">Aside: why this module is in Python</span>

Python trades execution speed for development speed. It is interpreted and dynamically typed, which is why vectorisation gave such a large speed-up in the revision notebook: it hands the work to compiled C code.

That trade-off is also why compiled or JIT-compiled languages (C++, Fortran, and increasingly Rust and Julia) still matter for large-scale physics simulations.
***

## <span style="color:#B5121B">3. Algorithmic complexity: how your code scales</span>

A related question: not "is the calculation correct?" but "how does its run time grow with the size of the dataset?" (the algorithmic complexity from the lecture). As an example, take a set of star positions and compute the sum of all pairwise distances.

In [ ]:
import numpy as np

def naive_pairwise_distance_sum(points):
    """Sum of all pairwise distances - naive nested-loop version."""
    n = len(points)
    total = 0.0
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = points[i, 0] - points[j, 0]
                dy = points[i, 1] - points[j, 1]
                total += np.sqrt(dx**2 + dy**2)
    return total

def vectorised_pairwise_distance_sum(points):
    """Same calculation, using NumPy broadcasting instead of Python loops."""
    diff = points[:, None, :] - points[None, :, :]
    dist = np.sqrt((diff**2).sum(axis=-1))
    return dist.sum()

# sanity check: both approaches should agree
rng = np.random.default_rng(0)
test_points = rng.uniform(0, 100, size=(50, 2))
a = naive_pairwise_distance_sum(test_points)
b = vectorised_pairwise_distance_sum(test_points)
print(f"naive result      : {a:.4f}")
print(f"vectorised result : {b:.4f}")
print(f"agree to within numerical precision: {np.isclose(a, b)}")

In [ ]:
import timeit

# Single timings are noisy (OS scheduling, caching, etc.) - take the minimum
# of several repeats for each n, which is the standard way to cut through that noise.
n_values = [50, 100, 200, 400, 800]
naive_times, vec_times = [], []

for n in n_values:
    points = rng.uniform(0, 100, size=(n, 2))

    tn = min(timeit.repeat(lambda: naive_pairwise_distance_sum(points), number=1, repeat=5))
    tv = min(timeit.repeat(lambda: vectorised_pairwise_distance_sum(points), number=1, repeat=5))

    naive_times.append(tn)
    vec_times.append(tv)
    print(f"n={n:4d}   naive={tn:8.4f}s   vectorised={tv:8.5f}s   ratio={tn/tv:6.1f}x")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(n_values, naive_times, "o-", color="firebrick", label="naive nested loop")
ax.plot(n_values, vec_times, "o-", color="dodgerblue", label="vectorised (NumPy)")
ax.set_xlabel("Number of points, $n$")
ax.set_ylabel("Time (s)")
ax.set_title("Pairwise-distance calculation: runtime vs. dataset size")
ax.legend()
fig.tight_layout()
plt.show()

The naive version grows roughly as $n^2$: doubling $n$ roughly quadruples the number of pairs, which is why the red curve rises so much more steeply than the blue one. The vectorised version scales much more gently over this range, because the work happens in compiled code operating on whole arrays. Datasets are now routinely thousands of rows long, and at that size the naive version becomes impractical.

> **Exercise:** add a third implementation using `scipy.spatial.distance.pdist` to the timing comparison and the plot. How does it compare with the vectorised version, and can you explain why?
***

## <span style="color:#FD9029">Independent exercise: nuclear binding energy</span>

The examples above were constructed. This one uses real published data. Work through it in your own time.

The mass of a nucleus is *less* than the total mass of its separate protons and neutrons. The missing mass, converted via $E=mc^2$, is the **binding energy** that holds the nucleus together. For a nuclide with $Z$ protons, $N$ neutrons (mass number $A = Z+N$) and measured atomic mass $M$:

$$BE = \left[Z \cdot m(^1\mathrm{H}) + N \cdot m_n - M\right] \times 931.494\ \mathrm{MeV/u}$$

The atomic mass of hydrogen-1 is used (rather than the proton mass) so that the electron masses approximately cancel. `isotope_masses.csv` gives $Z$, $N$, $A$ and the atomic mass (in u) for eight isotopes from deuterium to uranium, from NIST's Atomic Weights and Isotopic Compositions table.

In [ ]:
import pandas as pd
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week1/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename


m_H1 = 1.00782503223   # atomic mass of hydrogen-1, u (NIST)
m_n = 1.00866491606    # neutron mass, u (NIST CODATA)
u_to_MeV = 931.49410372  # 1 u in MeV (NIST CODATA)

isotopes = pd.read_csv(data_path("isotope_masses.csv"))
isotopes

**Part A.** Compute the binding energy per nucleon, $BE/A$, for each isotope in double precision (the default) and plot it against $A$. You should recover the nuclear binding energy curve, peaking around iron/nickel. This is why fusion releases energy for light elements and fission for heavy ones.

In [ ]:
Z = isotopes["Z"].to_numpy(dtype=np.float64)
N = isotopes["N"].to_numpy(dtype=np.float64)
A = isotopes["A"].to_numpy(dtype=np.float64)
M = isotopes["atomic_mass_u"].to_numpy(dtype=np.float64)

BE = (Z * m_H1 + N * m_n - M) * u_to_MeV
BE_per_A = BE / A

for name, a_val, bea in zip(isotopes["isotope"], A, BE_per_A):
    print(f"{name:18s}  A={a_val:6.0f}   BE/A = {bea:.4f} MeV")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(A, BE_per_A, "o-", color="darkslateblue")
ax.set_xlabel("Mass number, $A$")
ax.set_ylabel("Binding energy per nucleon (MeV)")
ax.set_title("The nuclear binding energy curve, from raw NIST mass data")
fig.tight_layout()
plt.show()

**Part B.** Repeat the calculation with every mass cast to `np.float16` (half precision, about 3 significant figures). Plot both curves on the same axes. The masses go up to 238 and the physics is in the 4th–5th significant figure: what do you expect *before* you run it?

In [ ]:
Z16 = isotopes["Z"].to_numpy(dtype=np.float16)
N16 = isotopes["N"].to_numpy(dtype=np.float16)
A16 = isotopes["A"].to_numpy(dtype=np.float16)
M16 = isotopes["atomic_mass_u"].to_numpy(dtype=np.float16)

BE16 = (Z16 * np.float16(m_H1) + N16 * np.float16(m_n) - M16) * np.float16(u_to_MeV)
BE_per_A16 = BE16 / A16

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(A, BE_per_A, "o-", color="darkslateblue", label="float64 (correct)")
ax.plot(A, BE_per_A16, "o--", color="firebrick", label="float16")
ax.set_xlabel("Mass number, $A$")
ax.set_ylabel("Binding energy per nucleon (MeV)")
ax.set_title("The same calculation, cast to half precision")
ax.legend()
fig.tight_layout()
plt.show()

**Part C (harder).** The float16 error does not grow smoothly with $A$. For each isotope, compute the *fractional mass defect*, $(Z\,m(^1\mathrm{H}) + N\,m_n - M) / M$: how small the difference is relative to the numbers being subtracted. Does it predict which isotopes have the worst float16 error? What does this tell you about when a precision choice that "should be fine" (238 fits easily in float16's range) isn't?

**Part D (reflection, no code).** In one or two sentences: if a nuclear-physics code computed binding energies in single precision (`float32`) rather than double, would you fix it? Use your results from Parts B and C to justify your answer.

***
## Summary

Two lessons: 
1. A numerically "correct" formula can still be badly wrong in practice
2. The same calculation can scale very differently depending on how it's written 

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>